In [1]:
import pandas as pd
df = pd.read_csv('data/updated_densitymap_hourly_cleaned.csv')

In [3]:
df.head()

,date,hour,building_floor,building_desc,latitude,longitude,building_type,density_cnt,capacity,dc_ratio
0,2021-03-31,1,4,209 BAY STATE ROAD,42.3507,-71.1028,Residential,1,8,0.13
1,2021-03-31,1,2,200-202 BAY STATE ROAD,42.3503,-71.1026,Residential,1,12,0.08
2,2021-03-31,1,2,184 BAY STATE ROAD,42.3503,-71.1019,Residential,1,6,0.17
3,2021-03-31,1,3,190 BAY STATE ROAD,42.3503,-71.1021,Residential,1,9,0.11
4,2021-03-31,1,3,161 BAY STATE ROAD,42.3506,-71.1009,Residential,1,5,0.20


In [4]:
bottle_fill = pd.read_csv('data/bottle_fillers.csv')
bottle_fill.head()

,Address A,Address B,Floor,Description,Accessibility,Handicap Accessible,Space@Bu room,Quantity,Date Installed,Type,Walk-Thru Notes,Completed
0,100,Ashford St,First,"Hallway across from vending machines, across f...",Public,Yes,191,1.0,NaN,Combo Bottle Filler/Drinking Fountain,two additional drinking fountain locations in ...,1/5/2024
1,120,Ashford St,First,"Kitchenette, Room 116A",Restricted,Yes,116A,1.0,NaN,Office-Style Water Cooler,NaN,1/5/2024
2,120,Ashford St,Second,"Kitchenette, Room 222",Restricted,Yes,222,1.0,NaN,Office-Style Water Cooler,NaN,1/5/2024
3,8,St. Mary's Street,Basement,Hallway by bathrooms,NaN,NaN,B81,1.0,Dec-23,Combo Bottle Filler/Drinking Fountain,NaN,NaN
4,8,St. Mary's Street,First,Hallway by electrical room,NaN,NaN,181,1.0,NaN,NaN,NaN,NaN


In [5]:
# Remove addresses that are nan

df = df.dropna(subset=['building_desc'])
bottle_fill = bottle_fill.dropna(subset=['Address A', 'Address B'])

In [6]:
import re

def average_address_if_range(address):
    ''' Takes in a range of addresses and returns the average address
        Ex: 20 - 30 Buswell St -> 25 Buswell St
    '''
    regex = r"(\d+)\s*-\s*(\d+)(.*)"
    parts = re.fullmatch(regex, address)
    if parts == None:
        return address

    start = int(parts.group(1))
    end = int(parts.group(2))
    street = parts.group(3)

    new = (start + end) // 2
    return str(new) + street

In [7]:
from functools import reduce

density_addresses = set(df.iloc[:, 3])
bottle_fill_address = set(bottle_fill['Address A'] + ' ' + bottle_fill['Address B'])

apply_transforms = lambda addresses, fns: [reduce(lambda acc, fn: fn(acc), fns, adr) for adr in addresses]

append_city = lambda address: address + ", Boston, MA"
transforms = [average_address_if_range, append_city]

density_addresses = apply_transforms(density_addresses, transforms)
bottle_fill_address = apply_transforms(bottle_fill_address, transforms)

density_addresses = list(set(density_addresses))
bottle_fill_address = list(set(bottle_fill_address))

print(density_addresses)
print(bottle_fill_address)


['848 BEACON STREET, Boston, MA', '1019 COMMONWEALTH AVENUE, Boston, MA', '115 BAY STATE ROAD, Boston, MA', '75 CUMMINGTON MALL, Boston, MA', '595 COMMONWEALTH AVENUE, Boston, MA', '185 BAY STATE ROAD, Boston, MA', '96 MOUNTFORT STREET, Boston, MA', '75 BAY STATE ROAD, Boston, MA', '233 BAY STATE ROAD, Boston, MA', '60 BAY STATE ROAD, Boston, MA', '700 BEACON STREET, Boston, MA', '31 BUSWELL STREET, Boston, MA', '48 BUSWELL STREET, Boston, MA', "15 ST. MARY'S STREET, Boston, MA", '855 COMMONWEALTH AVENUE, Boston, MA', '871 BEACON STREET, Boston, MA', '110 BAY STATE ROAD, Boston, MA', '4 CUMMINGTON MALL, Boston, MA', '5 BUSWELL STREET, Boston, MA', '844 BEACON STREET, Boston, MA', '737 COMMONWEALTH AVENUE, Boston, MA', '22 BUSWELL STREET, Boston, MA', '483 COMMONWEALTH AVENUE, Boston, MA', '277 BABCOCK STREET, Boston, MA', '182 BAY STATE ROAD, Boston, MA', '25 ABERDEEN STREET, Boston, MA', '25 BUICK STREET, Boston, MA', '214 BAY STATE ROAD, Boston, MA', '273 BABCOCK STREET, Boston, MA',

In [8]:
from google_maps import get_distance_matrix_large

res = get_distance_matrix_large(density_addresses, bottle_fill_address)

KeyboardInterrupt: 

In [9]:
print(res)

NameError: name 'res' is not defined

In [28]:
import requests 
import os
import requests
from dotenv import load_dotenv

load_dotenv()
GOOGLE_MAPS_API = os.getenv('GOOGLE_MAPS')

MODE = 'walking'
origin = '91 Bay State Road, Boston, MA'
destination = '96 Mountfort Street, Boston, MA'

url = 'https://maps.googleapis.com/maps/api/distancematrix/json'
params = {
    'key': GOOGLE_MAPS_API,
    'origins': origin,
    'destinations': destination,
    'mode': MODE
}

res = request.get(url, params=params)

In [36]:
print(res.json())
body = res.json()
row = body['rows'][0]
element = row['elements'][0]
distance = element['duration']['value']
print(distance)

{'destination_addresses': ['96 Mountfort St, Boston, MA 02215, USA'], 'origin_addresses': ['91 Bay State Rd, Boston, MA 02215, USA'], 'rows': [{'elements': [{'distance': {'text': '1.0 km', 'value': 964}, 'duration': {'text': '14 mins', 'value': 839}, 'status': 'OK'}]}], 'status': 'OK'}
839


In [ ]:
s1 = 'abcd'
s2 = '1234'


